# **PS-06****  
**

# **Problem Set 6: Project Genesis – The Chaos Engine (Week 6)**

  

## **📖 The Story: Entering the Chamber of Chaos**

In the deterministic domains of physical laws and neural network layers, you successfully taught the Oracle to internalize rigid, continuous geometric structures. But the real world flows through chaotic currents of probability, market fluctuations, and stochastic noise. Welcome to the **Chamber of Chaos**.

The old-world sorcerers of the MATLAB Citadel simulated random variables using stateful, single-threaded pseudo-random loops. If they wanted to evaluate a million market trajectories, their screens would freeze for hours under the weight of sequential CPU loops. We do not accept these limitations.

Equipped with [Antigravity Skills](https://antigravity.google/docs/skills), your development environment is no longer a passive text editor; it is an active, agent-driven ecosystem capable of writing code, analyzing execution traces, and running massively parallel stochastic pipelines autonomously. Today, you will inherit this power to conquer randomness itself.

## **🧠 Theoretical Deep Dive: Randomness under the Hood**

Before writing code, you must understand the mathematical framework of Monte Carlo methods and why traditional random generation fails on modern accelerator hardware (GPUs/TPUs).

### **1. The Geometry of Pi Estimation**

The purest distillation of Monte Carlo integration is the geometric estimation of  $\pi$ . Imagine a quadrant of a circle with radius  $r = 1$  inscribed perfectly within a unit square of side length  $1$ .

The area of the unit square is  $A_{\text{square}} = 1^2 = 1$ , while the area of the quarter circle is:

 $A_{\text{quarter}} = \frac{\pi r^2}{4} = \frac{\pi}{4}$ 

If we uniformly sample  $N$  independent random coordinate pairs  $(x, y)$  inside the square, the probability  $P$  that a point falls inside the quarter circle corresponds exactly to the ratio of their areas:

 $P = \frac{A_{\text{quarter}}}{A_{\text{square}}} = \frac{\pi}{4}$ 

By tracking how many points satisfy the algebraic condition  $x^2 + y^2 \le 1$ , we can approximate the value of  $\pi$  through simple frequency scaling:

 $\frac{N_{\text{inside}}}{N_{\text{total}}} \approx \frac{\pi}{4} \implies \pi \approx 4 \times \frac{N_{\text{inside}}}{N_{\text{total}}}$ 

### **2. The Mathematics of High-Dimensional Monte Carlo Integration**

Moving past basic geometry, we can scale this concept to compute complex, definite multi-dimensional integrals over a domain  $\Omega \subset \mathbb{R}^d$ :

 $I = \int_{\Omega} g(x) \, dx$ 

Analytical integration becomes intractable when  $g(x)$  is highly non-linear or when the dimensionality  $d$  is large. Classical numerical integration techniques (like Simpson’s rule or the Trapezoidal rule) suffer heavily from the **Curse of Dimensionality**: their error scales as  $\mathcal{O}(N^{-k/d})$ , meaning the number of grid points  $N$  required to maintain accuracy grows exponentially with  $d$ .

Monte Carlo bypasses this curse by reframing the integration problem as an expected value. If we introduce a probability density function  $p(x)$  that is uniform over the domain  $\Omega$ , the integral can be rewritten as:

 $I = V \int_{\Omega} g(x) p(x) \, dx = V \cdot \mathbb{E}[g(X)]$ 

where  $V = \int_{\Omega} 1 \, dx$  is the total volume of the domain, and  $X$  is a random variable distributed uniformly over  $\Omega$ . By the **Strong Law of Large Numbers**, the empirical sample mean converges to the true expected value as the number of independent and identically distributed (i.i.d.) samples  $N$  approaches infinity:

 $\hat{I}_N = \frac{V}{N} \sum_{i=1}^{N} g(X_i) \xrightarrow{p} I$ 

According to the **Central Limit Theorem**, the statistical error of this estimation drops at a rate of:

 $\text{Error} \sim \mathcal{O}\left(\frac{\sigma}{\sqrt{N}}\right)$ 

where  $\sigma$  is the standard deviation of  $g(X)$ . **Crucially, this convergence rate is completely independent of the dimension**  $d$ , making Monte Carlo methods the gold standard for complex engineering and high-dimensional system simulations.

### **3. Why JAX? Stateless Randomness & Mass Vectorization**

Traditional python frameworks (like standard NumPy or native Python) handle random number generation via a **global mutable state**. When you call numpy.random.rand(), the system reads an internal seed state, updates it via a stateful generator, and returns the random number:

 $\text{State}_{t+1}, X = \text{NextRandom}(\text{State}_t)$ 

In highly parallelized, asynchronous execution architectures like GPUs and TPUs, **global mutable states are an absolute disaster**. They introduce race conditions, ruin determinism, and prevent compiler optimizations like parallel mapping.

**JAX solves this by enforcing functional purity through stateless PRNGs** based on the counter-based *Threefry* algorithm. In JAX, the random state is an explicit, immutable array called a PRNGKey.

1.  To generate random numbers, you must pass the key explicitly to the generator function.
2.  The generator consumes the key and outputs the random numbers. It does *not* mutate the key.
3.  If you need multiple independent random variables or consecutive execution steps, you must explicitly fork the key using jax.random.split():

  

import jax  
  
key = jax.random.PRNGKey(42)  
key_distribution, key_agent = jax.random.split(key, 2)  
  

  

By enforcing this functional purity, JAX allows you to wrap your simulation logic inside jax.vmap (Vectorized Map). Instead of executing simulations sequentially using a slow Python for-loop, vmap automatically maps the computation across thousands of GPU/TPU hardware registers simultaneously, executing millions of stochastic paths in parallel at compiled C++ speeds.

## **🎮 Gamified Rules & Mechanics**

To navigate the random fluctuations of this week's assignment, your workspace tracks the following live game metrics:

1.  **The Luck Stat (PRNG Integrity):** If you pass a mutated, reused, or un-split PRNGKey into a function instead of utilizing stateless functional splits, your Luck instantly drops to 0, causing random parameter collisions and a penalty on your evaluation score.
2.  **The Swarm Multiplier:** Successfully orchestrating parallel worker subagents using the [Antigravity IDE](https://antigravity.google/) and utilizing specialized [Antigravity Skills](https://antigravity.google/docs/skills) unlocks the *Swarm Commander* badge, yielding a bonus multiplier on your execution logs.
3.  **Boss Encounter:** Exercise 4 pits your mathematical pipeline against a randomized macro-economic "Black Swan" anomaly. Your choice of algorithm determines your defensive baseline.


## **Exercise 1: The Antiquated Circle (Classical NumPy Pi Estimation)**

**Learning Objective:** Observing the limits of classical stateful randomness, measuring CPU overhead, and visualizing basic geometric convergence.

Before upgrading to accelerated clusters, you must experience the constraints of the old paradigm. You will write a standard Python script using standard numpy to estimate  $\pi$  via uniform spatial distribution.

### **Implementation Directives:**

1.  Create a script named src/classical_pi.py.
2.  Generate  $5,000,000$  random  $(x, y)$  coordinate points uniformly distributed between  $0$  and  $1$  using standard stateful numpy.random.uniform().
3.  Compute the Euclidean distance from the origin for every point ( $x^2 + y^2$ ). Count how many points fall inside the unit circle boundary.
4.  Calculate your empirical estimation of  $\pi$ . Measure the exact wall-clock execution time of this generation process using time.perf_counter().
5.  Extract a random subset of  $10,000$  points. Color-code them based on their position: blue for points inside the circle boundary, and red for points outside.

**Deliverables:** Run your file via uv run python src/classical_pi.py. Generate a scatter plot showing the quarter-circle boundary line and the distributed points. Save the image to data/classical_pi_disp.png. Record the execution time and your calculated value of  $\pi$  in your submission log.


In [ ]:
# Write your code for Exercise 1 here


## **Exercise 2: The Quantum Leap (The JAX Monte Carlo Engine)**

**Learning Objective:** Implementing a multi-variable Monte Carlo business simulation using stateless jax.random and compiling via jax.vmap.

Now, transition to the stateless paradigm. You will scale up from simple geometric figures to a business enterprise simulation, wrapping your logic inside pure functional components to run on hardware registers.

The annual net revenue function depends on three distinct stochastic variables:

1.  **Market Demand (** $D$ **):** Modeled as a Normal distribution:  $D \sim \mathcal{N}(\mu=1000, \sigma^2=150^2)$ 
2.  **Production Asset Cost (** $C$ **):** Modeled as a Log-Normal distribution to prevent negative costs and capture long-tailed expenses:  $\ln(C) \sim \mathcal{N}(\mu=5.5, \sigma^2=0.3^2)$ 
3.  **Regulatory Penalty Rate (** $R$ **):** Modeled as a Uniform distribution representing compliance fluctuations:  $R \sim \mathcal{U}(0.05, 0.25)$ 

The deterministic economic engine maps these variables to the net profit via the equation:

 $\text{Revenue} = (D \times 150.0) - C \times (1.0 - R)$ 

### **Implementation Directives:**

1.  In your workspace, create a script named src/monte_carlo.py.
2.  Write a mathematically pure JAX function simulate_path(key) that consumes a single subkey, splits it to isolate three distinct random samples for  $D$ ,  $C$ , and  $R$ , and computes the scalar net revenue.
3.  In your main execution block, initialize your master key: key = jax.random.PRNGKey(42).
4.  Do **not** write a Python loop. Use jax.random.split to generate  $1,000,000$  unique subkeys.
5.  Apply jax.vmap to your simulate_path function to evaluate all 1,000,000 keys in parallel across the hardware.
6.  Calculate the expected value  $\mathbb{E}[\text{Revenue}]$  and the Value-at-Risk ( $VaR_{95\%}$ ) threshold (the 5th percentile of the sorted revenue array).

**Deliverables:** Run your file using your environment tooling (uv run python src/monte_carlo.py). Generate a clean distribution histogram using matplotlib or plotly. Draw a solid black vertical line indicating the expected revenue, and a red dashed vertical line representing the  $VaR_{95\%}$  threshold. Save the resulting graphic as data/revenue_dist.png.


In [ ]:
# Write your code for Exercise 2 here


## **Exercise 3: Agentic Automation via Antigravity Skills**

**Learning Objective:** Programmatically configuring and unleashing autonomous subagent workflows using the [Antigravity IDE](https://antigravity.google/) and [Antigravity Skills](https://antigravity.google/docs/skills).

Instead of manually editing scripts, running benchmarks, and fixing syntax errors yourself, you will delegate these tasks to a coordinated group of subagents operating inside the secure, cross-platform sandbox of the Antigravity ecosystem.

                       $$ Gemini 3.5 Flash Orchestrator $$  
                                       |  
        +----------------------+----------------------+  
         |                                                                                       |  
$$ Subagent-Alpha $$                                          $$ Subagent-Beta $$  
 (Stress-Tester)                                                           (Profiler)  
         |                                                                                        |  
 Modifies cash flow variances                  Tracks JAX TPS & compilation  
  

### **The Swarm Directive:**

1.  Launch your workspace using the [Antigravity IDE](https://antigravity.google/).
2.  Review the workspace capability framework via [Antigravity Skills](https://antigravity.google/docs/skills) to understand how the agent can interact with file operations, process execution, and local metrics tracking.
3.  Configure your master orchestration layer to utilize **Gemini 3.5 Flash** for high-speed long-context processing. Provide the agent with the following system prompt:

"Observer-Prime, initiate an automated task sweep inside the Antigravity IDE sandbox. Utilize your filesystem and bash skills to evaluate our src/monte_carlo.py pipeline.

  - Spawn **Subagent-Alpha ('The Stress-Tester')**: This subagent must programmatically alter the variance parameters ( $\sigma$ ) of the Log-Normal asset cost distribution in src/monte_carlo.py to find the breaking point where the  $VaR_{95\%}$  drops below zero.
  - Spawn **Subagent-Beta ('The Profiler')**: This subagent must execute the script sequentially twice and extract execution times to profile the exact performance overhead of the initial JAX compilation trace versus the second warm execution pass.  
    Synthesize all agentic findings into a unified report named docs/Swarm_Stress_Report.md."

**Deliverables:** Monitor the live execution block as the subagents utilize their workspace skills to read, modify, and execute your code. Submit the generated markdown document (docs/Swarm_Stress_Report.md) containing the analytical outputs of both automated subagents.


In [ ]:
# Write your code for Exercise 3 here


## **Exercise 4: Boss Fight – Defeating the Black Swan (Markov Chains)**

**Learning Objective:** Implementing sequential state transitions using Markov Chains and handling runtime injection anomalies via jax.lax.scan.

The financial stability of your deep-tech enterprise does not happen in isolation. The macro-economy perpetually shifts between states of existence, which can be modeled as a discrete-time Markov Chain.

### **⚔️ Choose Your Weapon (Gamified Choice Node):**

Before implementing your code, declare in your documentation which algorithmic strategy module you will deploy to track system states over a continuous timeline of 365 simulated days:

  - **Module Alpha (The Matrix Carrier):** You build a full transition probability matrix and track a single aggregate probability state vector over time using a highly optimized functional loop driven by jax.lax.scan.
  - **Module Beta (The Individualist Swarm):** You instantiate 100,000 independent agent entities, evaluating their individual step-by-step state transitions completely in parallel via vectorized random sampling.

### **The Macro-Economic Architecture:**

The system shifts between three discrete conditions: **$$State 0: Bull Market, State 1: Stagnation, State 2: Catastrophic Recession$$**. The baseline transition probability matrix is defined as:

 $P = \begin{pmatrix} 0.85 & 0.12 & 0.03 \\ 0.10 & 0.75 & 0.15 \\ 0.05 & 0.20 & 0.75 \end{pmatrix}$ 

### **The Black Swan Shock Injection:**

1.  Create src/markov_boss.py and implement your chosen strategy module using pure, immutable JAX operations.
2.  **The Sabotage:** At day 180, an unexpected global economic crisis strikes the simulation. For exactly 10 days (from day 180 to day 190), completely overwrite the transition row variables. Force the probability vectors of both State 0 and State 1 to shift 80% of their transition probability mass directly into State 2 (Catastrophic Recession).
3.  After day 190, restore the baseline matrix variables to evaluate the system recovery rate.

**Deliverables:** Execute your model pipeline. Export a graph showing the percentage distribution of the three macro states across the full 365-day timeline. In your summary, write a 3-sentence paragraph detailing how your enterprise cash-flow structure from Exercise 2 would collapse if the random inputs were actively tied to this volatile Markov environment during the recession spike.


In [ ]:
# Write your code for Exercise 4 here


## **Exercise 5: Vault Ascension & Live Deployment**

A modern system engineer seals their codebase within a secure repository before presenting the live outputs to the world.

Open your terminal interface and use uv add to lock down your dependencies:  
  

  
uv add plotly openpyxl matplotlib  
  

  

1.  Commit your newly generated assets (src/classical_pi.py, src/monte_carlo.py, src/markov_boss.py, docs/Swarm_Stress_Report.md, and the generated images) to your local repository.
2.  Use the [Antigravity IDE](https://antigravity.google/) git skill set to push the latest codebase changes safely onto your remote repository branch.
3.  Update your automated project documentation site (index.md, README, or GitHub Pages) to showcase the revenue distribution graphs, the subagent workflow logs, and your Markovian defense metrics.

**Deliverables:** Provide your live public repository URL and a direct hyperlink to your updated project showcase page. The Chaos Engine is now fully operational under agentic control\!

  
**  
**

In [ ]:
# Write your code for Exercise 5 here
